In [1]:
import openai
import json
from utils import Spotify
import tiktoken
import time

In [2]:
# Load the tokenizer for GPT-4 (gpt-4 and gpt-3.5-turbo use the same encoding)
encoding = tiktoken.encoding_for_model("gpt-4o")
with open('../../config.json') as config_file:
    
    config = json.load(config_file)

In [21]:
client_id = config['client_id']
client_secret = config['client_secret']
redirect_uri = config['redirect_uri']
# username = 'byv1tdsf0wr3gpo2hkjkfd0tk'
sp = Spotify(client_id, client_secret, redirect_uri)
sp.connect()
playlist_df = sp.get_playlist()
sp.get_tracks_from_playlists()
df = sp.get_df()

In [ ]:
# !rm .cache

In [22]:
# col_name_dict = {
#     'danceability': 'dance'
#     'acousticness': 'acoustic',
#     'instrumentalness': 'instrum',
#     # 'duration_ms': 'duration',
#     # 'time_signature': 'timesig',
#     # 'speechiness': 'speech',
#     # 'loudness': 'loud',

# }
def count_tokens(messages):
    total_tokens = 0
    for message in messages:
        # Role adds an extra token for each message
        role_token_count = len(encoding.encode(message['role']))
        # Content token count
        content_token_count = len(encoding.encode(message['content']))
        # Add both role and content tokens to total count
        total_tokens += role_token_count + content_token_count + 2  # +2 for separators (message overhead)
    return total_tokens

In [7]:
# Things to do to reduce tokens
## Round decimals. So they might have similar tokens
## Feature selection
## Change feature names. i.e. accoustincess to accoustic
## Batching:
##    - For each iteration, identify tracks that might be a part of the playlist, and aggregate the result

# Things to add
## A temperature (0 - 1) mark to allow users to tell how closely related should a track be to be a part of the playlist.
## For example: If temperature is closer to 1 then more tracks will be added as the model is allowed to be more creative.
##              If temperature is closer to 0, then less tracks will be added as model less creative.
## Distribution or skewness of temperature??

In [25]:
# Based on playlist name
from sklearn.utils import shuffle

def generate_playlist_from_name(playlist_name, temperature, df):
    openai.api_key = config['api_key']
    system_content = """
    You are a spotify music playlist curator who organizes tracks into thematic playlists based on their audio features extracted
    from spotify api. 
    - The inputs will be given in the following format:
      playlist name: ...
      temperature: ...
      data: ...
    - The data in csv format will be supplied and the features below will be given for each track. These features can be helpful 
      in clustering the tracks into playlists but not they need not be used. 
    - The playlist name will be the theme of the playlist. Based on the playlist name, extract tracks in the data that matches
    the theme and respond with the list of tracks that should belong in the playlist.
    - The temperature parameter is similar to that of the temperature parameter for LLMs. It ranges from 0 to 1, where 1 is creative and 0 is not creative.
      This parameter tells the model how creative it should be when determining if a track belongs to the playlist theme. Naturally, higher temperature should
      mean more tracks are added to the playlist. 
    - The features in the data will be:
        Danceability: Measures how suitable a track is for dancing, based on rhythm and tempo. (Range: 0 to 1)
        Energy: Intensity and activity of a track. Higher values represent more energetic tracks. (Range: 0 to 1)
        Acousticness: Likelihood that a track is acoustic. (Range: 0 to 1)
        Instrumentalness: Measures the likelihood of no vocals. Higher means more instrumental. (Range: 0 to 1)
        Valence: Positiveness or happiness of a track. Higher values sound more positive. (Range: 0 to 1)
        Loudness: Average volume in decibels (dB). (Range: ~ -60 to 0 dB)
        Tempo: Speed of the track, measured in beats per minute (BPM). (Range: 0 to 300+ BPM)

    The response should only be the list of song separated by comma. i.e. songA, songB, songC, ... and so on. Note that songs needs to be related to the theme.
    For example, if there is no afro beats songs in the list of songs given, then no songs can be returned. In the case where no songs are returned, return an empty string.
    """
    prompt = f"""
    playlist name: {playlist_name}
    temperature: {temperature}
    data: {df.to_string(na_rep='NA')}
    """
    messages=[
        {"role": "system", "content": system_content},
        {"role": "user", "content": prompt},
    ]
    # print(prompt)
    print(f"Number of tokens number {len(encoding.encode(system_content)) + len(encoding.encode(prompt))}")
    print(f"Number of tokens: {count_tokens(messages)}")
    try:
        response = openai.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            temperature= 0.7
        )
    except openai.RateLimitError:
        raise Exception("Please try again in a few minutes. If this problem persists, contact ...")
        
    return response.choices[0].message.content
df = sp.get_df()
df = shuffle(df)
df = sp.get_df()
df = df.loc[5:150, ~df.columns.isin(['playlist', 'id'])]
start = time.monotonic()
string = generate_playlist_from_name('Morning drive', 0.5, df)
print(string)
print(f"Time taken: {round(time.monotonic()-start, 4)} seconds")

Number of tokens number 7310
Number of tokens: 7316
Morning Papers, Morning Jazz Juice, Morning Jazz Juice, Coffee House Bebop Jazz Background, A Book and a Glass of Wine, Jazz Coffee Vibes, Work Coffee Jazz, Lazy Morning Jazz, Jazz Relaxing Mornings
Time taken: 2.3573 seconds


In [60]:
df = sp.get_df()
df.columns

Index(['name', 'danceability', 'energy', 'acousticness', 'instrumentalness',
       'valence', 'loudness', 'tempo'],
      dtype='object')

In [19]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth

# Set up Spotipy credentials
# SPOTIFY_CLIENT_ID = 'your_client_id'  # Replace with your Spotify client ID
# SPOTIFY_CLIENT_SECRET = 'your_client_secret'  # Replace with your Spotify client secret
SPOTIFY_REDIRECT_URI = 'http://localhost:8080/callback'  # This should match your app's redirect URI

# Authenticate and get token
scope = "playlist-modify-public"
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=client_id,
                                               client_secret=client_secret,
                                               redirect_uri=SPOTIFY_REDIRECT_URI,
                                               scope=scope))

# User information
user_id = sp.current_user()["id"]

def create_playlist(playlist_name, song_list):
    # Step 1: Create a public playlist
    playlist = sp.user_playlist_create(user=user_id, name=playlist_name, public=True)
    playlist_id = playlist['id']

    # Step 2: Search for each song and add to the playlist
    track_ids = []
    for song in song_list:
        results = sp.search(q=song, type='track', limit=1)
        if results['tracks']['items']:
            track_ids.append(results['tracks']['items'][0]['id'])

    if track_ids:
        sp.playlist_add_items(playlist_id, track_ids)
        print(f"Playlist '{playlist_name}' created with {len(track_ids)} songs!")
    else:
        print("No valid tracks found to add to the playlist.")

if __name__ == "__main__":
    # Input: List of songs
    song_string = "I KNOW ?, BUTTERFLY EFFECT, all of me, n.h.i.e., Sprinter, Glock In My Lap, née-nah, redrum"
    songs = [song.strip() for song in song_string.split(",")]

    # Create a playlist
    playlist_name = "My New Playlist"  # Change this to your preferred playlist name
    create_playlist(playlist_name, songs)


Playlist 'My New Playlist' created with 8 songs!


In [ ]:
import tiktoken

# Load the tokenizer for GPT-4 (gpt-4 and gpt-3.5-turbo use the same encoding)
encoding = tiktoken.encoding_for_model("gpt-4o")

# Text you want to tokenize
text = "Track 1 details: upbeat and energetic."

# Tokenize the text
tokens = encoding.encode('..')
len(tokens)

1